In [ ]:
from pathlib import Path
import glob
import iris
import numpy as np
import importlib

import functions
importlib.reload(functions)
from functions import *

data_dir = "/scratch/hydro4/users/kv25483/FDRI/Rain_gauge_optimisation/Data/"
catchment = gpd.read_file(data_dir + "54080/54080.shp")
ceh_gear_dir = "/scratch/hydro4/shared_data/climate_observed/CEH-GEAR/hourly/1km/"

In [ ]:
import xarray as xr

def iris_to_xarray(cube):
    data = cube.data

    coords = {}
    dims = []

    for dim_coord in cube.dim_coords:
        name = dim_coord.name()
        coords[name] = dim_coord.points
        dims.append(name)

    return xr.DataArray(data, coords=coords, dims=dims, name=cube.name())

In [ ]:
from pathlib import Path
import xarray as xr
data_dir = "/scratch/hydro4/users/kv25483/FDRI/Rain_gauge_optimisation/Data/"
catchment_Dolwen = gpd.read_file(data_dir + "54080/54080.shp")

HAD_names = sorted(Path(data_dir + "HAD_rainfall_annual/").glob("*.nc"))
HAD_layers = [xr.open_dataset(f) for f in HAD_names]
HAD_ann = xr.concat([ds['rainfall'] for ds in HAD_layers], dim="time")

HAD_ann = HAD_ann.rio.write_crs("EPSG:27700")
HAD_Dolwen_ppn = HAD_ann.rio.clip(catchment_Dolwen.geometry, crs=catchment_Dolwen.crs,drop=True, all_touched=True)
HAD_Dolwen_ppn_mean = HAD_Dolwen_ppn.mean(dim="time")

In [ ]:
# ---------------------------
# 1. Load CEH-GEAR (Iris)
# ---------------------------
filenames = []

for year in range(1990, 2017):
    general_filename = ceh_gear_dir + f'CEH-GEAR-1hr-v2_{year}*'
    filenames.extend(glob.glob(general_filename))

print("Number of files:", len(filenames))

# Load cubes
cubes = iris.load(filenames)

# Extract rainfall variable
rain_cubes = iris.cube.CubeList([c for c in cubes if c.name() == 'rainfall_amount'])

# Remove conflicting attributes (important for concatenation)
for c in rain_cubes:
    c.attributes = {}

# Concatenate into one cube
ceh_gear = rain_cubes.concatenate_cube()

# ---------------------------
# 2. Trim to catchment bbox
# ---------------------------
ceh_gear_trimmed = trim_to_bbox_of_region_obs(ceh_gear,catchment,'projection_y_coordinate','projection_x_coordinate',1)

# ---------------------------
# 3. Convert to xarray
# ---------------------------
# ceh_xr = ceh_gear_trimmed.to_xarray()
ceh_xr = iris_to_xarray(ceh_gear_trimmed)

# ---------------------------
# 4. Fix Y-axis orientation
# (CEH often runs north → south)
# ---------------------------
y_name = "projection_y_coordinate"
x_name = "projection_x_coordinate"

if ceh_xr[y_name][0] > ceh_xr[y_name][-1]:
    ceh_xr = ceh_xr.sortby(y_name)

# ---------------------------
# 5. Regrid onto HAD grid
# ---------------------------
ceh_on_had = ceh_xr.interp( {x_name: HAD_Dolwen_ppn_mean[x_name],  y_name: HAD_Dolwen_ppn_mean[y_name]}, method="linear")

# ---------------------------
# 6. Done — CEH now matches HAD grid spatially
# ---------------------------
print(ceh_on_had)

In [ ]:
import numpy as np

def describe_grid(cube, name):
    x = cube.coord('projection_x_coordinate')
    y = cube.coord('projection_y_coordinate')
    
    print(f"\n--- {name} ---")
    print("Shape:", cube.shape)
    print("X range:", x.points.min(), "→", x.points.max())
    print("Y range:", y.points.min(), "→", y.points.max())
    
    # Resolution (assumes regular grid)
    dx = np.diff(x.points).mean()
    dy = np.diff(y.points).mean()
    print("Resolution dx:", dx)
    print("Resolution dy:", dy)

def describe_xarray_grid(da, name):
    print(f"\n--- {name} ---")
    print("Shape:", da.shape)
    
    print("X range:", da.projection_x_coordinate.min().values, "→", da.projection_x_coordinate.max().values)
    print("Y range:", da.projection_y_coordinate.min().values, "→", da.projection_y_coordinate.max().values)
    
    dx = float(da.projection_x_coordinate.diff("projection_x_coordinate").mean())
    dy = float(da.projection_y_coordinate.diff("projection_y_coordinate").mean())
    print("Resolution dx:", dx)
    print("Resolution dy:", dy)

describe_xarray_grid(HAD_Dolwen_ppn_mean, "HAD (xarray)")    
describe_xarray_grid(ceh_on_had, "CEH (ON HAD) (xarray)")    
describe_grid(ceh_gear_trimmed, "CEH-GEAR")


In [ ]:
import rioxarray
ceh_on_had = ceh_on_had.rio.write_crs("EPSG:27700")
ceh_on_had.to_netcdf("Data/ceh_on_had.nc")